# TrustLens — Phase H: Relationship & Network Intelligence

**Deterministic Cross-Modal Relationship Graph Construction**

This notebook connects independently observed marketplace attributes (Phases A–G.1) into an explicit, deterministic NetworkX relationship graph. Every edge is traceable to its forensic evidence source.

### Strict Scientific Guardrails
- **NO fraud scores**, scam probabilities, or risk indexes.
- **NO seller profiling** or common ownership inferences.
- **Overconnection Prevention**: Components represent cohesive observational reuse, separating high-density generic visual candidates.
- **Traceability**: All relationships explicitly track their underlying evidence source.

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path

from trustlens.marketplace.network_intelligence import NetworkIntelligenceEngine
print("Libraries and NetworkIntelligenceEngine imported successfully!")

## 1. Load Frozen Artifacts (Phases A–G.1)
Initialize engine and inspect frozen datasets.

In [ ]:
engine = NetworkIntelligenceEngine()
engine.load_frozen_datasets()
print(f"Listings count: {len(engine.listings_df):,}")
print(f"Media fingerprints: {len(engine.fingerprints_df):,}")
print(f"Image relationships (Phase C): {len(engine.image_rel_df):,}")
print(f"Deep visual relationships (Phase D): {len(engine.deep_rel_df):,}")
print(f"Text similarity candidates (Phase F): {len(engine.text_sim_df):,}")
print(f"Image OCR records (Phase E): {len(engine.ocr_df):,}")

## 2. Entity Construction
Construct deterministic entity nodes for Listings, Media, Products, Cities, and States.

In [ ]:
nodes = engine.build_entity_nodes()
node_types = pd.Series([n["node_type"] for n in nodes.values()]).value_counts()
print(f"Total entity nodes: {len(nodes):,}")
print(node_types)

## 3. Relationship Edge Construction
Construct controlled taxonomy edges across image, text, OCR, and containment layers.

In [ ]:
edges = engine.build_relationship_edges(nodes)
edges_df = pd.DataFrame(edges)
print(f"Total relationship edges: {len(edges_df):,}")
print(edges_df["relationship_type"].value_counts())

## 4. Connected Components & Overconnection Prevention
Analyze connected components formed by direct observational reuse relationships.

In [ ]:
components, feat_df = engine.extract_connected_components(nodes, edges, include_visual_similarity=False)
comp_df = pd.DataFrame(components)
print(f"Total connected components: {len(comp_df):,}")
non_single = comp_df[comp_df["node_count"] > 1]
print(f"Non-singleton components: {len(non_single):,}")
print(f"Largest component size: {int(comp_df.iloc[0].node_count)} listings")
comp_df[["component_id", "node_count", "city_count", "state_count", "product_family_count", "exact_image_edges", "text_similarity_edges"]].head(10)

## 5. Multi-Signal Relationship Candidates
Identify pairs connected across $\ge 2$ independent evidence layers without scoring fraud.

In [ ]:
multi_signal = engine.extract_multi_signal_candidates(edges)
print(f"Total multi-signal pairs: {len(multi_signal):,}")
multi_df = pd.DataFrame(multi_signal)
print(multi_df["evidence_count"].value_counts().sort_index(ascending=False))
multi_df[["listing_a", "listing_b", "evidence_count", "relationship_types", "cross_city", "cross_state"]].head(10)

## 6. Geographic Corridors Analysis
Descriptive analysis of observable cross-city and cross-state relationship corridors.

In [ ]:
cross_city_edges = edges_df[edges_df["cross_city"]]
print(f"Total cross-city edges: {len(cross_city_edges):,}")
corridors = cross_city_edges.apply(lambda r: " <-> ".join(sorted([str(r["city_a"]), str(r["city_b"])])), axis=1).value_counts().head(10)
print("Top 10 Cross-City Corridors:")
print(corridors)

## 7. Exported Parquet Tables Verification
Verify exported Parquet tables and integrity.

In [ ]:
p_edges = pd.read_parquet("data/olx_processed/relationship_edges.parquet")
p_comp = pd.read_parquet("data/olx_processed/relationship_components.parquet")
p_feat = pd.read_parquet("data/olx_processed/relationship_features.parquet")
print(f"Exported relationship_edges: {len(p_edges):,} rows")
print(f"Exported relationship_components: {len(p_comp):,} rows")
print(f"Exported relationship_features: {len(p_feat):,} rows")